# Exploration des instances OR-Library (Beasley)

Inventaire de toutes les instances MKP dans `data/knapsack/beasley/`.

**Deux formats coexistent :**
- **mknap1 / mknapcb\*** — format standard OR-Library : `K` instances, chacune avec header `n m opt`
- **mknap2** — format annoté : instances labellisées `problem NAME.DAT`, header `n_knapsacks m_objects` (sans opt dans le header, opt à la fin du bloc)

In [1]:
import re
from pathlib import Path

import pandas as pd

BEASLEY_DIR = Path("data/knapsack/beasley")

## Parsers

In [2]:
def _numeric_tokens(filepath):
    """Yield tokens numériques uniquement (strip commentaires // et texte)."""
    pat = re.compile(r'^-?[0-9]+(\.[0-9]*)?([eE][+-]?[0-9]+)?$')
    with open(filepath) as f:
        for line in f:
            for tok in line.split('//', 1)[0].split():
                if pat.match(tok):
                    yield tok


def parse_standard(filepath):
    """Parse mknap1.txt et mknapcb*.txt.

    Format par instance :
        n  m  opt
        profits[n]
        A[m, n]          (m contraintes × n items)
        capacities[m]
    """
    gen = _numeric_tokens(filepath)
    stem = Path(filepath).stem
    K = int(next(gen))
    records = []
    for k in range(K):
        n   = int(next(gen))
        m   = int(next(gen))
        opt = float(next(gen))
        for _ in range(n + m * n + m):  # profits + matrice + capacités
            next(gen)
        records.append(dict(file=stem, idx=k + 1, name=f"{stem}_{k+1}", n=n, m=m, opt=opt))
    return records


def parse_mknap2(filepath):
    """Parse mknap2.txt.

    Instances séparées par des balises 'problem NAME.DAT'.
    Format par instance :
        n_knapsacks  m_objects
        profits[m]
        capacities[n]
        A[n, m]          (n contraintes × m items)
        opt
    """
    # Extraire les blocs par nom de problème
    problem_re = re.compile(r'^\s*problem\s+(\S+)', re.IGNORECASE)
    numeric_re = re.compile(r'^-?[0-9]+(\.[0-9]*)?([eE][+-]?[0-9]+)?$')

    blocks = []   # list of (name, [numeric_tokens])
    current_name = None
    current_tokens = []

    with open(filepath) as f:
        for line in f:
            m = problem_re.match(line)
            if m:
                if current_name is not None:
                    blocks.append((current_name, current_tokens))
                current_name = m.group(1).replace('.DAT', '').replace('.dat', '')
                current_tokens = []
            elif current_name is not None:
                for tok in line.split('//', 1)[0].split():
                    if numeric_re.match(tok):
                        current_tokens.append(tok)
    if current_name is not None:
        blocks.append((current_name, current_tokens))

    stem = Path(filepath).stem
    records = []
    for idx, (name, tokens) in enumerate(blocks, 1):
        gen = iter(tokens)
        n   = int(next(gen))    # n_knapsacks
        m   = int(next(gen))    # m_objects
        # Consommer : profits[m] + capacities[n] + A[n*m] + opt
        skip = m + n + n * m
        for _ in range(skip):
            next(gen)
        opt = float(next(gen))
        records.append(dict(
            file=stem, idx=idx,
            name=name,
            n=m,    # convention : n = #items, m = #contraintes
            m=n,
            opt=opt,
        ))
    return records

## Lecture de tous les fichiers

In [3]:
all_records = []

for fp in sorted(BEASLEY_DIR.glob("*.txt")):
    try:
        if fp.name == "mknap2.txt":
            records = parse_mknap2(fp)
        else:
            records = parse_standard(fp)
        all_records.extend(records)
        print(f"{fp.name:20s}  {len(records):3d} instances  "
              f"n∈[{min(r['n'] for r in records)},{max(r['n'] for r in records)}]  "
              f"m∈[{min(r['m'] for r in records)},{max(r['m'] for r in records)}]")
    except Exception as e:
        print(f"{fp.name:20s}  ERREUR: {e}")

df = pd.DataFrame(all_records)
print(f"\nTotal : {len(df)} instances dans {df['file'].nunique()} fichiers")

mknap1.txt              7 instances  n∈[6,50]  m∈[5,10]
mknap2.txt             48 instances  n∈[20,105]  m∈[2,30]
mknapcb1.txt           30 instances  n∈[100,100]  m∈[5,5]
mknapcb2.txt           30 instances  n∈[250,250]  m∈[5,5]
mknapcb3.txt           30 instances  n∈[500,500]  m∈[5,5]

Total : 145 instances dans 5 fichiers


## Vue complète

In [4]:
pd.set_option('display.max_rows', 200)
df[['file', 'idx', 'name', 'n', 'm', 'opt']]

,file,idx,name,n,m,opt
0,mknap1,1,mknap1_1,6,10,3800.0
1,mknap1,2,mknap1_2,10,10,8706.1
2,mknap1,3,mknap1_3,15,10,4015.0
3,mknap1,4,mknap1_4,20,10,6120.0
4,mknap1,5,mknap1_5,28,10,12400.0
5,mknap1,6,mknap1_6,39,5,10618.0
6,mknap1,7,mknap1_7,50,5,16537.0
7,mknap2,1,SENTO1,60,30,7772.0
8,mknap2,2,SENTO2,60,30,8722.0
9,mknap2,3,WEING1,28,2,141278.0


## Résumé par fichier

In [5]:
(
    df.groupby('file')
    .agg(
        instances=('name', 'count'),
        n_min=('n', 'min'), n_max=('n', 'max'),
        m_min=('m', 'min'), m_max=('m', 'max'),
        opt_min=('opt', 'min'), opt_max=('opt', 'max'),
    )
    .reset_index()
)

,file,instances,n_min,n_max,m_min,m_max,opt_min,opt_max
0,mknap1,7,6,50,5,10,3800.0,16537.0
1,mknap2,48,20,105,2,30,776.0,1095445.0
2,mknapcb1,30,100,100,5,5,0.0,0.0
3,mknapcb2,30,250,250,5,5,0.0,0.0
4,mknapcb3,30,500,500,5,5,0.0,0.0


## Paires (n, m) distinctes

In [6]:
(
    df.groupby(['n', 'm'])
    .agg(count=('name', 'count'), fichiers=('file', lambda s: ', '.join(sorted(s.unique()))))
    .reset_index()
    .sort_values(['n', 'm'])
)

,n,m,count,fichiers
0,6,10,1,mknap1
1,10,10,1,mknap1
2,15,10,1,mknap1
3,20,10,2,"mknap1, mknap2"
4,27,4,1,mknap2
5,28,2,6,mknap2
6,28,4,1,mknap2
7,28,10,1,mknap1
8,29,2,1,mknap2
9,30,5,5,mknap2
